> Os exemplos desta página usam dados sintéticos gerados pelo projeto. Não são observações ASTER nem permitem interpretar valores como reflectância, temperatura ou classes reais. Veja [os dados de exemplo](../../../../data/README.md).



# Capítulo 06 — Classificação Supervisionada (Parte 1)

> 📖 **Referência:** Canty (2019), Cap. 6  
> Notas: [`aulas/classificadores_supervisionados.md`](../aulas/classificadores_supervisionados.md)  
> Código completo do livro: `legado/canty/auxil/scripts/classify.py`

---

## 1. Intuição: ML vs classificação em sensoriamento remoto

Em *machine learning* genérico, classificamos vetores de atributos com rótulos. Em **sensoriamento remoto**, o atributo costuma ser o **espectro do pixel** (e índices derivados), e o rótulo é uma **classe de cobertura**.

O Cap. 6 enfatiza o classificador de **máxima verossimilhança gaussiana** (Bayes com $p(\mathbf{x}|k)$ normal multivariada): estime média e covariância por classe nas amostras de treino e atribua cada pixel à classe de maior densidade a posteriori.

Redes neurais relaxam a hipótese gaussiana; o treino está no [Apêndice B](../../treinamento-de-redes/aulas/treino_redes.md).

## 2. Fundamentos

$$
\hat{k}(\mathbf{x}) = \arg\max_k \; p(\mathbf{x}|k)\,P(k)
$$

Com $p(\mathbf{x}|k)=\mathcal{N}(\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k)$, isso equivale (em log) a minimizar a distância de Mahalanobis mais $\tfrac12\log|\boldsymbol{\Sigma}_k|$ (e o prior).

No sklearn: `GaussianNB` (covariâncias diagonais) e `QuadraticDiscriminantAnalysis` (QDA, $\boldsymbol{\Sigma}_k$ cheia).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import rasterio

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_utils").is_dir())
EXEMPLO = REPO / 'data/exemplos/exemplo_sintetico.tif'
OUT = REPO / "conteudo/classificacao/supervisionada/figuras"
OUT.mkdir(exist_ok=True)
print(EXEMPLO.exists(), EXEMPLO)

## 3. Implementação — rótulos sintéticos via NDVI e QDA/NB

In [ ]:
with rasterio.open(EXEMPLO) as ds:
    img = ds.read().astype(np.float64)
bands, rows, cols = img.shape
ndvi = (img[2] - img[1]) / (img[2] + img[1] + 1e-6)
y = np.zeros(ndvi.shape, dtype=int)
y[ndvi >= 0.35] = 2
y[(ndvi >= 0.1) & (ndvi < 0.35)] = 1

X = img.reshape(bands, -1).T
y_flat = y.ravel()
rng = np.random.default_rng(0)
idx = rng.choice(len(y_flat), size=min(12000, len(y_flat)), replace=False)
X_s, y_s = X[idx], y_flat[idx]

Xtr, Xte, ytr, yte = train_test_split(X_s, y_s, test_size=0.3, stratify=y_s, random_state=0)
sc = StandardScaler().fit(Xtr)
clf = QuadraticDiscriminantAnalysis().fit(sc.transform(Xtr), ytr)
yp = clf.predict(sc.transform(Xte))
print('acurácia', accuracy_score(yte, yp))
print(confusion_matrix(yte, yp))

## 4. Visualização

In [ ]:
pred = clf.predict(sc.transform(X)).reshape(rows, cols)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(ndvi, cmap='RdYlGn'); ax[0].set_title('NDVI-like')
ax[1].imshow(y, cmap='tab10'); ax[1].set_title('Rótulos sintéticos')
ax[2].imshow(pred, cmap='tab10'); ax[2].set_title('QDA')
for a in ax: a.axis('off')
fig.tight_layout()
fig.savefig(OUT / 'nb06_qda_map.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Próximos passos

- Rodar `python exercicios/ex06_1.py`.
- Comparar classificadores no Cap. 7.
- Substituir rótulos sintéticos por amostras reais (shapefile + `legado/canty/auxil/readshp.py`).